# Day 3: Supervised ML — Linear Regression

<a target="_blank" href="https://colab.research.google.com/github/LuWidme/uk259/blob/rework/demos/07_Regression.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏱️ **Time budget:** ~75 min · **Day 3** — Linear & multiple regression; statsmodels is bonus.


**Duration:** 1.5–2 hours  
**Prerequisites:** Python basics, NumPy, Pandas, data preprocessing  

**Learning Objectives:**
- Understand what linear regression is and when to use it
- Build **simple** and **multiple** regression models with scikit-learn
- Evaluate models using MAE, RMSE, and R²
- Analyze residuals to check model assumptions
- Understand Simpson's Paradox

**Datasets Used:** Penguins (seaborn), Melbourne Housing (`datasets/melb_data.csv`), Tips (seaborn, bonus)

> **Bonus:** An optional section at the end covers statistical inference with **statsmodels** (p-values, OLS summary) for those who want to go deeper.

---


In [ ]:
# === COURSE SETUP — run this cell first! ===
# Installs the required packages (most are preinstalled on Google Colab)
# and downloads the course datasets so the notebook works out of the box.
%pip install -q numpy pandas matplotlib seaborn scikit-learn

import os, urllib.request

DATA_URL = "https://raw.githubusercontent.com/LuWidme/uk259/rework/datasets/"
for folder in ("datasets", os.path.join("..", "datasets")):
    os.makedirs(folder, exist_ok=True)
    for fname in ['titanic.csv', 'melb_data.csv', 'Company_data.csv']:
        path = os.path.join(folder, fname)
        if not os.path.exists(path):
            urllib.request.urlretrieve(DATA_URL + fname, path)

print("Setup complete — you are ready to go!")

## What is Linear Regression?

Linear regression predicts a **continuous numerical value** (unlike classification, which predicts categories).

**Examples:**
- Predict **house price** from size, location, rooms
- Predict **tip amount** from bill total
- Predict **body mass** from flipper length

**How it works:** Find the "best-fit" line through the data.

**Simple linear regression** (one feature): `y = mx + b`
- `m` = slope (how much y changes when x increases by 1)
- `b` = intercept (value of y when x = 0)

**Multiple linear regression** (many features): `y = b₀ + b₁x₁ + b₂x₂ + ... + bₙxₙ`

![img](https://github.com/LuWidme/uk259/blob/rework/img/1_N1-K-A43_98pYZ27fnupDA%20(1).jpg?raw=1)

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme()
print("✓ Libraries imported successfully!")

---

## Part 1: Exploring the Data

We'll start with the **Palmer Penguins** dataset — measurements of penguins from three species.

In [ ]:
penguins = sns.load_dataset('penguins')
print(f"Dataset: {penguins.shape[0]} penguins, {penguins.shape[1]} columns")
print(f"Species: {list(penguins['species'].unique())}")
penguins.head()

In [ ]:
# Pairplot: visualize relationships between all numerical features
sns.pairplot(data=penguins, hue='species')
plt.suptitle('Penguin Feature Relationships by Species', y=1.02)
plt.show()

### Task 1: Visualize a Linear Relationship

Use `sns.lmplot` to show the correlation between `bill_length_mm` (x-axis) and `bill_depth_mm` (y-axis). The function automatically fits a regression line.

**Hint:** `sns.lmplot(data=penguins, x='bill_length_mm', y='bill_depth_mm')`

Do you notice anything surprising about the trend?

In [ ]:
# TODO: Plot bill_length_mm vs bill_depth_mm with a regression line
# Hint: sns.lmplot(data=penguins, x='...', y='...')

g = None  # TODO: create the lmplot

plt.show()

### Simpson's Paradox

The overall trend shows a **negative** correlation (longer bill → shallower depth). But is this real?

This is **Simpson's Paradox**: a trend in aggregated data can **reverse** when you look at subgroups.

Plot the data again, but split it by species. use the "hue" parameter of lm plot to fit separate regression lines for each species:

In [ ]:
g = sns.lmplot(
    data=penguins,
    x='bill_length_mm',
    y='bill_depth_mm',
    hue='species',
    height=6,
    aspect=1.5
)

g.set_axis_labels('Bill length (mm)', 'Bill depth (mm)')
plt.suptitle("Simpson's Paradox: Trend Reverses Within Species!", y=1.02)
plt.show()


---

## Part 2: Linear Regression with Scikit-Learn

### The Idea Behind Linear Regression

Imagine you have a scatterplot of data points — for example, penguin flipper length on the x-axis and body mass on the y-axis. You can see that bigger flippers tend to go with heavier penguins. Linear regression **draws the best possible straight line** through those points.

But what makes a line "the best"? The algorithm tries every possible line and picks the one that **minimizes the total error** — the distances between the line and the actual data points.

### How It Finds the Best Line

For each data point, the **residual** is the vertical distance between the point and the line:

```
residual = actual value − predicted value
```

Linear regression minimizes the **sum of squared residuals** (this is why it's also called **Ordinary Least Squares / OLS**):

```
minimize:  Σ (actual - predicted)²
```

Why squared? Squaring makes all errors positive (so negative and positive errors don't cancel out) and penalizes large errors more heavily than small ones.

The result is an equation like:

```
body_mass = 49 × flipper_length − 5600
```

This tells us: "For every additional millimeter of flipper length, body mass increases by about 49 grams."

### The Scikit-Learn Workflow

Scikit-learn uses a consistent 3-step pattern for all models:

1. **Create** the model: `model = LinearRegression()`
2. **Train** (fit) it on data: `model.fit(X_train, y_train)`
3. **Predict** new values: `y_pred = model.predict(X_test)`

Let's predict penguin **body mass** from **flipper length**.

In [ ]:
# Prepare data
penguins_clean = penguins.dropna(subset=['flipper_length_mm', 'body_mass_g'])

X = penguins_clean[['flipper_length_mm']].values  # Features (must be 2D)
y = penguins_clean['body_mass_g'].values           # Target

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

print(f"\nLearned equation:")
print(f"  body_mass = {model.coef_[0]:.2f} × flipper_length + {model.intercept_:.2f}")
print(f"\nInterpretation: For every 1mm increase in flipper length,")
print(f"  body mass increases by ~{model.coef_[0]:.0f}g")

In [ ]:
# Visualize predictions
y_pred = model.predict(X_test)

plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, alpha=0.5, label='Training data', color='blue')
plt.scatter(X_test, y_test, alpha=0.5, label='Test data', color='green')

# Regression line
X_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(X_line, model.predict(X_line), 'r-', linewidth=2, label='Regression line')

plt.xlabel('Flipper Length (mm)')
plt.ylabel('Body Mass (g)')
plt.title('Linear Regression: Predicting Penguin Body Mass')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

## Part 3: Evaluation Metrics

How do we know if our model is good? We need quantitative metrics.

| Metric | What it measures | Interpretation |
|--------|-----------------|----------------|
| **MAE** | Average absolute error | "On average, we're off by X grams" |
| **RMSE** | Root of average squared error | Penalizes large errors more |
| **R²** | Proportion of variance explained | 1.0 = perfect, 0 = useless |

In [ ]:
# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
r2_train = r2_score(y_train, model.predict(X_train))

print("=" * 50)
print("MODEL EVALUATION")
print("=" * 50)
print(f"\nMean Absolute Error (MAE):  {mae:.2f} g")
print(f"Root Mean Squared Error:    {rmse:.2f} g")
print(f"R² Score (test):            {r2:.4f}")
print(f"R² Score (train):           {r2_train:.4f}")

print(f"\nInterpretation:")
print(f"- The model explains {r2*100:.1f}% of the variance in body mass")
print(f"- Predictions are off by ~{mae:.0f}g on average")
if abs(r2_train - r2) < 0.05:
    print("- Small train/test gap → no overfitting")

---

## Part 4: Residual Analysis

### Why We Check Residuals

Getting a number like "R² = 0.78" tells you *how much* variance the model explains, but it doesn't tell you *whether the model is appropriate for the data*. A linear model can have a decent R² and still be fundamentally wrong — for example, if the true relationship is curved, a straight line will systematically over-predict in some regions and under-predict in others.

**Residuals** are the tool we use to catch these problems. A residual is simply:

```
residual = actual value − predicted value
```

If the model is a good fit, the residuals should look like **random noise** — no patterns, no trends, roughly the same spread everywhere. If you see a pattern in the residuals, it means the model is missing something.

### What to Look For

We'll create four diagnostic plots. Here's what a **healthy** model looks like in each:

| Plot | What you want to see | Red flag |
|------|---------------------|----------|
| **Residuals vs Predicted** | Random scatter around y=0 | A curve or funnel shape |
| **Histogram of Residuals** | Bell-shaped (normal distribution) | Strong skew or multiple peaks |
| **Q-Q Plot** | Points following the diagonal line | Points curving away at the ends |
| **Actual vs Predicted** | Points hugging the 45° line | Systematic drift away from the line |

If the residuals show clear patterns, it often means:
- The relationship isn't linear (try polynomial regression)
- An important feature is missing from the model
- The data has outliers that are pulling the line

In [ ]:
from scipy import stats

residuals = y_test - y_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Residuals vs Predicted
axes[0, 0].scatter(y_pred, residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Predicted')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Histogram of Residuals
axes[0, 1].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Residuals')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Residuals')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Actual vs Predicted
axes[1, 1].scatter(y_test, y_pred, alpha=0.6)
axes[1, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
                'r--', lw=2, label='Perfect prediction')
axes[1, 1].set_xlabel('Actual Values')
axes[1, 1].set_ylabel('Predicted Values')
axes[1, 1].set_title('Actual vs Predicted')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("How to read these plots:")
print("1. Residuals vs Predicted: Points should be randomly scattered around y=0")
print("2. Histogram: Should look like a bell curve (normal distribution)")
print("3. Q-Q Plot: Points should follow the diagonal line")
print("4. Actual vs Predicted: Points should follow the red dashed line")

---

## Part 5: Multiple Regression

### From One Feature to Many

So far we predicted body mass using only **flipper length** — a single number. But real outcomes usually depend on **many factors at once**. A house's price depends on its size, location, number of rooms, and age; a penguin's body mass depends on more than just its flippers.

**Multiple regression** extends the same "best-fit line" idea to several features at once:

```
y = b0 + b1*x1 + b2*x2 + b3*x3 + ...
```

Each coefficient tells you: **"how much does y change when this feature increases by 1, while the others stay the same?"**

The scikit-learn workflow is *exactly the same* as before — the only difference is that `X` now has several columns instead of one. Let's predict penguin body mass from three measurements and see if it beats the single-feature model.

In [ ]:
# Multiple features: flipper length + bill length + bill depth
features = ['flipper_length_mm', 'bill_length_mm', 'bill_depth_mm']
penguins_multi = penguins.dropna(subset=features + ['body_mass_g'])

X_multi = penguins_multi[features].values
y_multi = penguins_multi['body_mass_g'].values

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

model_multi = LinearRegression()
model_multi.fit(X_train_m, y_train_m)
y_pred_m = model_multi.predict(X_test_m)

r2_multi = r2_score(y_test_m, y_pred_m)
mae_multi = mean_absolute_error(y_test_m, y_pred_m)

print("Multiple regression (3 features):")
print(f"  R2  = {r2_multi:.4f}")
print(f"  MAE = {mae_multi:.1f} g")
print(f"\nSingle-feature model (Parts 2-3) had R2 = {r2:.4f}")
print(f"Improvement in R2: {r2_multi - r2:+.4f}")

print("\nLearned coefficients (effect of +1 unit, holding others constant):")
for feat, coef in zip(features, model_multi.coef_):
    print(f"  {feat:20s}: {coef:+.2f} g")

---

## Part 6: Exercises

### Exercise 1: Melbourne Housing Price Prediction

**Scenario:** Predict house prices using the Melbourne housing dataset.

**Tasks:**
1. Load `datasets/melb_data.csv` and explore it
2. Select numerical features and handle missing values
3. Build a linear regression model with scikit-learn
4. Evaluate using MAE, RMSE, and R²
5. Create residual plots

**Hints:**
- Good features to start with: `Rooms`, `Distance`, `Bathroom`, `Landsize`, `BuildingArea`
- Use `df.dropna()` or `df.fillna(df.median())` for missing values
- Target variable: `Price`

In [ ]:
# Step 1: Load and explore the Melbourne housing data
melb = pd.read_csv('../datasets/melb_data.csv')
print(f"Dataset: {melb.shape[0]} houses, {melb.shape[1]} columns")
print(f"\nColumns: {list(melb.columns)}")
print(f"\nMissing values (top 5):")
print(melb.isnull().sum().sort_values(ascending=False).head())
melb.head()

In [ ]:
# Step 2: Select features and prepare data

# TODO: Choose features for the model
feature_cols = None  # TODO: e.g., ['Rooms', 'Distance', 'Bathroom', 'Landsize', 'BuildingArea']

# TODO: Create feature matrix X and target vector y
# Hint: Drop rows where any selected feature or Price is NaN
X_melb = None  # TODO
y_melb = None  # TODO: melb['Price'] (after dropping NaN)

# TODO: Split into train/test
# Hint: train_test_split(X_melb, y_melb, test_size=0.2, random_state=42)

In [ ]:
# Step 3 & 4: Train model and evaluate

# TODO: Create and train a LinearRegression model

# TODO: Make predictions on test set

# TODO: Calculate and print MAE, RMSE, R²

# TODO: Print the coefficients and their feature names
# Hint: model.coef_ gives the coefficients, pair with feature_cols

In [ ]:
# Step 5: Create residual plots

# TODO: Calculate residuals (y_test - y_pred)
# TODO: Create at least 2 diagnostic plots:
#   - Residuals vs Predicted
#   - Actual vs Predicted
# Do the residuals look random? Any patterns?

---

## Summary

In this notebook you learned:

✓ What linear regression is and how to interpret the equation  
✓ Simpson's Paradox — aggregated trends can be misleading  
✓ Building **simple** regression models with scikit-learn  
✓ Evaluation metrics: MAE, RMSE, R²  
✓ Residual analysis to check model assumptions  
✓ **Multiple** regression — using several features at once  

### Key Takeaways

1. **Always visualize first** — check for linearity before fitting a line
2. **Simpson's Paradox** — always explore subgroups in your data
3. **R² is not everything** — check residual plots too
4. **More features can help** — but compare against the simpler model
5. **Match the tool to the question** — scikit-learn for prediction, statsmodels (bonus) for statistical inference

### What's Next?

- **Polynomial regression:** Fit curves, not just lines
- **Regularization:** Ridge and Lasso regression to prevent overfitting
- **Neural Networks:** Learn complex, non-linear patterns

---

---

## Bonus: Statistical Inference with Statsmodels (optional)

*This section is optional — for learners who finished the core material and want to understand not just "how well does the model predict?" but "which features actually matter?"*

### Scikit-learn vs. Statsmodels

Both libraries fit a linear regression, but they answer different questions:

| | **Scikit-learn** | **Statsmodels** |
|---|---|---|
| **Focus** | Prediction accuracy | Statistical inference |
| **Gives you** | Coefficients, predictions | Coefficients, **p-values**, confidence intervals, R² |
| **Key question** | "How well does the model predict?" | "Which features actually matter?" |

The **p-value** for each feature answers: "Is this feature's effect real, or could it be due to random chance?" A p-value below 0.05 means we're fairly confident the feature has a genuine effect.

We'll use the **Tips** dataset to predict tip amount from bill total, party size, and smoking status.

In [ ]:
from statsmodels.formula.api import ols

tips = sns.load_dataset('tips')
print(f"Tips dataset: {tips.shape[0]} meals")
tips.head()

In [ ]:
# Fit a multiple regression model
formula = 'tip ~ total_bill + smoker + size'
result = ols(formula=formula, data=tips).fit()
result.summary()

### Reading the OLS Summary

The summary above looks intimidating, but you only need to focus on a few key areas. Let's walk through them one by one using the output we just produced.

---

#### 1. Model Fit (top-right block)

| Value | Our Result | What it means |
|-------|-----------|---------------|
| **R-squared** | 0.469 | The model explains 46.9% of the variance in tips |
| **Adj. R-squared** | 0.462 | Same, but penalizes adding useless features. Use this when comparing models with different numbers of features |
| **F-statistic** | 70.57 | Tests whether the model as a whole is useful. A large F-statistic (with a tiny Prob) means: "yes, at least one feature matters" |
| **No. Observations** | 244 | Number of data points used |

**Rule of thumb:** R² of 0.47 means the model captures roughly half the story — there are other factors influencing tips (service quality, mood, etc.) that we haven't measured.

---

#### 2. Coefficients Table (the most important part)

Each row represents one feature (or the intercept). Here's what every column means:

| Column | Meaning |
|--------|---------|
| **coef** | How much `tip` changes when this feature increases by 1 (holding all others constant) |
| **std err** | Uncertainty in the coefficient — smaller is better |
| **t** | `coef / std err` — how many standard errors the coefficient is away from zero |
| **P>\|t\|** | **The p-value.** The probability that this coefficient is actually zero (i.e. the feature has no real effect). Smaller = more confident the effect is real |
| **[0.025  0.975]** | 95% confidence interval — the true coefficient likely falls in this range |

**Reading our results row by row:**

| Feature | coef | P>\|t\| | Interpretation |
|---------|------|---------|----------------|
| **Intercept** | 0.63 | 0.003 | Baseline tip of ~€0.63 when all features are zero |
| **smoker[T.No]** | 0.08 | **0.546** | Non-smokers tip €0.08 more — but p = 0.55 means this could easily be random noise. **Not significant!** |
| **total_bill** | 0.094 | 0.000 | For every €1 increase in the bill, the tip goes up by ~€0.09. Highly significant (p ≈ 0) |
| **size** | 0.18 | 0.041 | Each additional person adds ~€0.18 to the tip. Significant (p < 0.05), but just barely |

---

#### 3. The Decision: Keep or Drop?

The p-value threshold is conventionally **0.05**:
- **p < 0.05** — the feature's effect is statistically significant. Keep it.
- **p > 0.05** — we can't be confident the effect is real. Consider dropping it.

In our model, `smoker` has p = 0.546 — far above 0.05. This means whether someone smokes or not doesn't meaningfully predict their tip amount in this dataset. We could simplify the model by removing it.

---

#### 4. Confidence Intervals — A Practical Check

Look at the `[0.025  0.975]` columns for `smoker[T.No]`: the interval is **[-0.188, 0.355]**. It crosses zero — meaning the true effect could be negative, zero, or positive. When a confidence interval includes zero, the feature is not significant (this always matches p > 0.05).

Compare with `total_bill`: its interval is **[0.076, 0.112]**. It's entirely positive — we're confident the effect is real and positive.

---

#### 5. Bottom Block (advanced — optional)

| Value | What it means |
|-------|---------------|
| **Omnibus / Prob(Omnibus)** | Tests if residuals are normally distributed. Prob < 0.05 means they're not perfectly normal (common in practice) |
| **Durbin-Watson** | Tests for autocorrelation in residuals. Values near 2.0 are good (ours is 2.1) |
| **Skew / Kurtosis** | Shape of residual distribution. Skew ≈ 0 and Kurtosis ≈ 3 is ideal |
| **Cond. No.** | Checks for multicollinearity (features that are too correlated). Values > 1000 are concerning |

You don't need to memorize these — just know they exist for when you need to dig deeper into model diagnostics.

### Visualizing the Simple Regression Line

In [ ]:
# Simple regression: tip ~ total_bill
result_simple = ols(formula='tip ~ total_bill', data=tips).fit()

m = result_simple.params['total_bill']
c = result_simple.params['Intercept']

plt.figure(figsize=(10, 6))
plt.scatter(tips['total_bill'], tips['tip'], alpha=0.5)
x_line = np.linspace(tips['total_bill'].min(), tips['total_bill'].max(), 100)
plt.plot(x_line, m * x_line + c, 'r-', linewidth=2, label=f'y = {m:.3f}x + {c:.3f}')
plt.xlabel('Total Bill (€)')
plt.ylabel('Tip (€)')
plt.title('Predicting Tips from Total Bill')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"R² = {result_simple.rsquared:.4f}")
print(f"For every €1 increase in bill, the tip increases by ~€{m:.2f}")

---

### Bonus Exercise: Feature Selection with Statsmodels

**Task:** Use statsmodels OLS to find which features are statistically significant predictors of tip amount.

1. Start with the formula: `tip ~ total_bill + size + smoker + sex + day + time`
2. Check p-values for each feature
3. Remove non-significant features (p > 0.05) one at a time
4. Compare R² of the full model vs. the simplified model

In [ ]:
# TODO: Fit the full model with all features
full_formula = None  # TODO: 'tip ~ total_bill + size + smoker + sex + day + time'

# TODO: result_full = ols(formula=full_formula, data=tips).fit()
# TODO: print(result_full.summary())

# TODO: Which features have p > 0.05? Remove them.

# TODO: Fit a simplified model without non-significant features
# TODO: Compare R² of full vs simplified model